# Math reference

Every equation and derivation used in this project, in the order it first
appears. The numbered notebooks give the intuition and the results. This
notebook gives the math step by step, so anyone can check it, and so I can
review it before an interview.

| Section | Topic | Used in |
|---|---|---|
| 1 | Normalized fares and log changes | 02 |
| 2 | Empirical conditional probability | 02 |
| 3 | Stopping times | 04 |
| 4 | Savings, and why the two percentages differ | 04, 05 |
| 5 | The hindsight ceiling | 04 |
| 6 | The Bellman equation | 04 |
| 7 | Proof that the best rule is a threshold | 04 |
| 8 | Choosing a parameter, and lookahead bias | 05 |
| 9 | Walk-forward validation | 05, 06 |
| 10 | Labels and right-censoring | 06 |
| 11 | The expected-value cutoff | 06 |
| 12 | Logistic regression | 06 |
| 13 | Gradient boosting | 06 |
| 14 | AUC and the Brier score | 06 |
| 15 | Calibration | 06 |
| 16 | Shapley values (SHAP) | 06 |
| 17 | The bootstrap and percentile confidence intervals | 09 |
| 18 | The cluster bootstrap | 09 |
| 19 | Paired comparison | 09 |
| 20 | Estimating a Markov chain | 07 |
| 21 | The jump model and its maximum-likelihood fit | 07 |
| 22 | Backward induction on a grid, and convergence | 07 |
| 23 | Monte Carlo error | 07 |
| 24 | Interpolation and kinks | 07 |
| 25 | The value of waiting (real options) | 07 |
| 26 | Fixed effects and the within transformation | 03 |
| 27 | Clustered standard errors | 03 |
| 28 | Regression discontinuity at the deadlines | 03 |
| 29 | Reading log coefficients, and the HHI | 03 |
| 30 | Autocorrelation and the variance ratio | 03 |
| 31 | Fitted Q-iteration | 08 |

**What kind of problem is this?** Buying a ticket is an **optimal stopping
problem**: a finite-horizon Markov decision process with two actions, buy or
wait, which is in turn a kind of stochastic optimization (minimize the
expected price paid). Because one traveler's purchase doesn't move prices,
any strategy can be replayed exactly on historical data. Notebook 04 explains
this framing in full.

**Notation used throughout**

| Symbol | Meaning |
|---|---|
| $i$ | a trip (one route on one flight date) |
| $t$ | days before departure |
| $P_t$ (or $P_{i,t}$) | cheapest fare $t$ days before departure |
| $s$ | the starting point: how many days out I start looking |
| $\tau$ | the day I buy |
| $S_i$ | savings on trip $i$ vs. buying at the start |

---

## 1. Normalized fares and log changes *(notebook 02)*

**Normalized fare.** To put a $120 route and a $600 route on one chart, I
divide each fare by that trip's median fare:

$$\text{rel}_{i,t} = \frac{P_{i,t}}{\operatorname{median}_t\, P_{i,t}}$$

A value of 1.0 means "this trip's usual price", and 1.5 means 50% above it.

**Log changes.** I measure moves as $\log(P_t / P_{t-1})$ rather than plain
percentages, because log changes **add up over time**. Using
$\log(a/b) = \log a - \log b$:

$$\log\frac{P_2}{P_0} = (\log P_2 - \log P_1) + (\log P_1 - \log P_0)
= \log\frac{P_2}{P_1} + \log\frac{P_1}{P_0}$$

Plain percentages don't add up: +10% then −10% gives $1.1 \times 0.9 = 0.99$,
a 1% loss. In logs, $+0.0953 - 0.1054 = -0.0101$, which sums correctly. Log
changes also treat rises and falls symmetrically, which is why quants use
them for asset returns.

---

## 2. Empirical conditional probability *(notebook 02)*

"How often does a cheaper fare show up later, when I'm $t$ days out?" is the
conditional probability

$$P\Big(\min_{u < t} P_u < P_t \;\Big|\; t \text{ days out}\Big)$$

I estimate it by counting: out of all trips observed at $t$ days out, the
share whose fare later went lower. With $n_t$ trips at day $t$:

$$\hat P = \frac{1}{n_t} \sum_{i} \mathbf{1}\Big[\min_{u<t} P_{i,u} < P_{i,t}\Big]$$

where $\mathbf{1}[\cdot]$ is 1 if the condition holds and 0 otherwise. This
uses future prices, so it describes what happened. It is not a strategy.

---

## 3. Stopping times *(notebook 04)*

Let $\mathcal{F}_t$ be everything I know on day $t$: today's fare, the fares
I've already seen, the route and the date. The day I buy, $\tau$, is a
**stopping time** if

$$\{\tau = t\} \text{ is determined by } \mathcal{F}_t \quad\text{for every } t$$

In words: whether I buy today can depend only on what I know today, never on
tomorrow's fare. This one condition separates a real strategy from cheating
with hindsight, and every strategy in the project satisfies it.

---

## 4. Savings, and why the two percentages differ *(notebooks 04, 05)*

Compared with buying at the start, a strategy saves

$$S_i = P_{i,s} - P_{i,\tau} \qquad\text{(negative means waiting cost me money)}$$

Let $r_i = S_i / P_{i,s}$ be trip $i$'s percent saved. I report two
percentages:

$$\text{average \%} = \frac{1}{n}\sum_i r_i \qquad\qquad
\text{total \%} = \frac{\sum_i S_i}{\sum_j P_{j,s}}$$

**Total % is a weighted average.** Multiply and divide each term by
$P_{i,s}$:

$$\text{total \%} = \sum_i \frac{P_{i,s}}{\sum_j P_{j,s}} \cdot \frac{S_i}{P_{i,s}}
= \sum_i w_i\, r_i, \qquad w_i = \frac{P_{i,s}}{\sum_j P_{j,s}}$$

So total % weights each trip by its fare, while average % weights every trip
equally ($1/n$).

**When they disagree.** Subtracting:

$$\text{total \%} - \text{average \%} = \sum_i \Big(w_i - \frac{1}{n}\Big)\, r_i$$

Trips with above-average fares have $w_i > 1/n$. So the difference is
positive exactly when **expensive trips save a larger share than cheap
ones**, that is, when % saved is positively correlated with the fare.
Notebook 05 found that: starting 60 days out, waiting until 21 days out saves
20% on tickets over $500 but 6% on tickets under $200.

---

## 5. The hindsight ceiling *(notebook 04)*

For any strategy and any price path, the price I pay is one of the prices on
that path, so it can't be below the path's minimum:

$$P_\tau \ge \min_{t \le s} P_t$$

Subtracting both sides from $P_s$ flips the inequality, and averaging over
trips keeps it:

$$\mathbb{E}[P_s - P_\tau] \;\le\; \mathbb{E}\Big[P_s - \min_{t \le s} P_t\Big]$$

This holds for every strategy, including the best possible one, so the
right-hand side (about 30% from 60 days out, 8% from 14) is a true upper bound
on what any strategy can save.

---

## 6. The Bellman equation *(notebook 04)*

Let $V_t(p)$ be the **expected price I'll end up paying** if today's fare is
$p$, there are $t$ days left, and I act optimally from here on.

**Step 1, the last day.** I must buy, so

$$V_1(p) = p$$

**Step 2, any earlier day.** If I wait, tomorrow I face the same problem with
$t-1$ days left and an unknown fare $P_{t-1}$. The expected cost of waiting is
the **continuation value**:

$$C_t(p) = \mathbb{E}\big[\,V_{t-1}(P_{t-1}) \mid P_t = p\,\big]$$

**Step 3, pick the cheaper option:**

$$V_t(p) = \min\big\{\, p,\; C_t(p) \,\big\}$$

Buy if $p \le C_t(p)$, otherwise wait. $V_t$ is defined through $V_{t-1}$, so
the equation is solved **backward** from departure: $V_1$, then $V_2$, and so
on. This is dynamic programming, and notebook 07 solves it numerically.

---

## 7. Proof that the best rule is a threshold *(notebook 04)*

**Claim.** The optimal rule has the form "buy if $P_t \le b_t$" for a
threshold $b_t$, and $b_t$ rises as departure approaches.

**Case 1: tomorrow's fare doesn't depend on today's.**

1. The conditional expectation in $C_t(p)$ then ignores $p$, so $C_t(p) = c_t$
   is a constant.
2. The rule "buy if $p \le c_t$" is a threshold, with $b_t = c_t$.
3. By definition, $c_{t+1} = \mathbb{E}[V_t(P_t)]$. Substituting
   $V_t(p) = \min(p, c_t)$ gives the recursion
   $$c_2 = \mathbb{E}[P_1], \qquad c_{t+1} = \mathbb{E}\big[\min(P_t,\ c_t)\big]$$
4. For any number $x$, $\min(x, c_t) \le c_t$. Expectations preserve
   inequalities, so $c_{t+1} \le c_t$, and therefore
   $$c_2 \ge c_3 \ge c_4 \ge \cdots$$
5. The more days left, the **lower** the threshold. Equivalently, the
   threshold rises as departure approaches: with fewer chances left for a
   better price, I should accept a higher one. This holds whatever the fare
   distributions are on each day. $\blacksquare$

**Case 2: today's fare matters (prices follow a Markov chain).**

1. I buy when $D(p) = p - C_t(p) \le 0$.
2. Suppose $C_t$ has slope less than 1: a fare that's $1 higher today raises my
   expected future cost by less than $1. That's the case when part of a price
   jump tends to reverse.
3. Then $D'(p) = 1 - C_t'(p) > 0$, so $D$ is increasing, and an increasing
   function crosses zero at most once.
4. So the buy region is $\{p \le b_t\}$, which is still a threshold.

Whether the thresholds also rise in Case 2 isn't guaranteed in general.
Notebook 07 checks it numerically on the estimated Markov chain.

**Interview version:** "Buying a ticket is an optimal stopping problem. The
Bellman equation compares buying now with the continuation value. If prices
are independent, the continuation value is a constant, so the optimal policy
is a reservation price. Since $\min(P, c) \le c$, that reservation price rises
as departure approaches."

---

## 8. Choosing a parameter, and lookahead bias *(notebook 05)*

The rule "wait until $N$ days out" has one parameter. The natural choice is
the $N$ with the highest average % saved:

$$N^* = \arg\max_N \; \frac{1}{n} \sum_i r_i(N)$$

If $N^*$ is chosen on the **same trips** it's evaluated on, the result is
biased upward. Among many candidates, the winner partly won by luck on those
particular trips, and that luck won't repeat on new trips. That's in-sample
overfitting. When the trips used come from the future relative to the
decision, it's also **lookahead bias**.

In notebook 05, choosing $N$ on all the data claims about 10% savings from 60
days out. Choosing it honestly (section 9) picks "buy now", which saves 0%.

---

## 9. Walk-forward validation *(notebooks 05, 06)*

For each test month $m$, the training set is

$$T_m = \{\text{trips } i : \text{flightDate}_i < \text{start}(m)\}$$

This is defined by **departure date, not search date**. A trip's outcome
(whether waiting would have paid) is only known once it has flown. A trip
searched before month $m$ but departing during it still has an unknown
outcome at the start of $m$, so using it would leak future information.

In notebook 06 there are also settings to choose (the label window $h$ and
the cutoff $\theta$). They're chosen on a **validation set**, the trips that
departed in the 4 weeks before $m$, using a model trained only on trips that
departed before that. Test month $m$ is never used to choose anything.

---

## 10. Labels and right-censoring *(notebook 06)*

For a decision $a$ days before departure, made on search date $d$, the label
is

$$y = \mathbf{1}\Big[\min_{\text{next } h \text{ days}} P \;\le\; (1 - 0.05)\, P_a\Big]$$

that is, did the fare drop at least 5% within the next $h$ days?

Data collection ended on October 5. If the next $h$ days run past that date,
part of the window is missing, and a drop after October 5 would look like
"no drop". This is **right-censoring**. A label is used only if its whole
window was collected:

$$d + \min(h,\ a - 1) \le \text{Oct 5}$$

(The $\min$ is there because the window can't extend past departure.)
Censored labels are marked unknown rather than guessed.

---

## 11. The expected-value cutoff *(notebook 06)*

Let $p$ be the predicted probability of a drop. If I wait, I gain about $G$
(as a share of the fare) when the drop comes, and lose about $L$ when it
doesn't. Waiting is worth it when the expected value is positive:

$$pG - (1-p)L > 0 \iff pG + pL > L \iff p > \frac{L}{G+L} = \theta$$

So "wait while $p > \theta$" **is** the expected-value rule. I tune $\theta$
directly on recent departed trips instead of estimating $G$ and $L$
separately.

**Reading a tuned $\theta$ backward:**

- $\theta = 0.5$ means $\frac{L}{G+L} = 0.5$, so $L = G$: gains and losses are
  about the same size.
- $\theta = 0.7$ means $L = \frac{0.7}{0.3}\,G \approx 2.3\,G$. A wrong wait
  costs about 2.3× what a right wait saves, so I need 70% confidence before
  waiting.

---

## 12. Logistic regression *(notebook 06)*

**Model.** A linear score $z = \beta^\top x$ is squashed into a probability by
the sigmoid:

$$p = \sigma(z) = \frac{1}{1 + e^{-z}}$$

Equivalently, the **log-odds** are linear: $\log\frac{p}{1-p} = \beta^\top x$.

**A useful derivative.** Differentiating $(1 + e^{-z})^{-1}$:

$$\sigma'(z) = \frac{e^{-z}}{(1+e^{-z})^2} = \sigma(z)\big(1 - \sigma(z)\big) = p(1-p)$$

**Fitting by maximum likelihood.** Each label $y_i \in \{0, 1\}$ has
probability $p_i^{y_i}(1 - p_i)^{1 - y_i}$. Maximizing the product over all
rows is the same as minimizing the negative log, the **log-loss**:

$$\mathcal{L}(\beta) = -\sum_i \Big[\, y_i \log p_i + (1 - y_i)\log(1 - p_i) \,\Big]$$

**Gradient.** For one row, the derivative of the loss with respect to $z_i$
is

$$\frac{\partial \ell_i}{\partial z_i}
= -\frac{y_i}{p_i}\, p_i(1-p_i) + \frac{1-y_i}{1-p_i}\, p_i(1-p_i)
= -y_i(1-p_i) + (1-y_i)p_i = p_i - y_i$$

and since $\partial z_i / \partial \beta = x_i$:

$$\nabla_\beta \mathcal{L} = \sum_i (p_i - y_i)\, x_i$$

Each row pushes $\beta$ in proportion to its error.

**Hessian (second derivative).**

$$H = \sum_i p_i(1 - p_i)\, x_i x_i^\top$$

Every term is positive semi-definite, since $p_i(1-p_i) \ge 0$. So the loss
is **convex**: there's one global minimum and no local traps.

**Solving.** Setting the gradient to zero has no closed-form solution, so it's
solved iteratively. Newton's method updates
$\beta \leftarrow \beta - H^{-1} \nabla\mathcal{L}$, and scikit-learn uses
L-BFGS, which approximates $H$ to save computation.

---

## 13. Gradient boosting (LightGBM) *(notebook 06)*

**Model.** The raw score is a sum of small decision trees,

$$F(x) = \sum_{m=1}^{M} \eta\, f_m(x)$$

with learning rate $\eta$, and $p = \sigma(F(x))$. Trees are added one at a
time, each correcting the current errors.

**Step 1: second-order Taylor expansion.** To add a new tree $f$ to the
current scores $F_i$, I want to minimize $\sum_i \ell(y_i, F_i + f(x_i))$.
Expanding each term around $F_i$:

$$\ell(y_i, F_i + f) \approx \ell(y_i, F_i) + g_i f + \tfrac{1}{2} h_i f^2,
\qquad g_i = \frac{\partial \ell}{\partial F_i}, \quad h_i = \frac{\partial^2 \ell}{\partial F_i^2}$$

For log-loss, section 12 gives $g_i = p_i - y_i$ and $h_i = p_i(1 - p_i)$.

**Step 2: one leaf.** A leaf $j$ outputs the same value $w$ for every row
$i \in I_j$ that lands in it. With an L2 penalty $\tfrac{1}{2}\lambda w^2$, the
part of the loss that depends on $w$ is

$$\sum_{i \in I_j}\Big(g_i w + \tfrac{1}{2} h_i w^2\Big) + \tfrac{1}{2}\lambda w^2
= G_j\, w + \tfrac{1}{2}(H_j + \lambda)\, w^2$$

where $G_j = \sum_{i \in I_j} g_i$ and $H_j = \sum_{i \in I_j} h_i$.

**Step 3: minimize the quadratic.** Setting the derivative to zero,
$G_j + (H_j + \lambda) w = 0$, so

$$w_j^* = -\frac{G_j}{H_j + \lambda}$$

That's a regularized **Newton step**, applied leaf by leaf.

**Step 4: choosing splits.** Plugging $w_j^*$ back in, the leaf's loss is
$-\dfrac{G_j^2}{2(H_j + \lambda)}$. A split is chosen to make the sum of this
quantity over the two new leaves as low as possible.

**Why it beat logistic regression.** A tree can split on one input and then
another, so it captures **interactions**, such as "a high fare matters more
far from departure". Logistic regression only adds effects together.

---

## 14. AUC and the Brier score *(notebook 06)*

**AUC.** Pick one random decision point where the fare dropped and one where
it didn't. AUC is the probability that the model gives the first one a
higher score:

$$\text{AUC} = P\big(\hat p_{\text{drop}} > \hat p_{\text{no drop}}\big)$$

- 0.5 means random ranking, and 1.0 means perfect ranking.
- It only measures **order**. A model can rank perfectly and still have
  badly calibrated probabilities.
- It equals the Mann–Whitney U statistic divided by the number of
  (drop, no-drop) pairs.

**Brier score.** The mean squared error of the probabilities:

$$\text{Brier} = \frac{1}{n}\sum_i (p_i - y_i)^2$$

Lower is better, and it penalizes both bad ranking and bad calibration. The
benchmark is a constant guess equal to the true rate $\bar y$. Its score is

$$\frac{1}{n}\sum_i (\bar y - y_i)^2 = \bar y (1 - \bar y)$$

which is about 0.24 when $\bar y \approx 0.4$. To be useful, a model has to beat
that.

---

## 15. Calibration *(notebook 06)*

A model is **calibrated** if, among all decision points where it predicts
probability $p$, the drop actually happens a share $p$ of the time:

$$P(y = 1 \mid \hat p = p) = p$$

To check it, I sort the predictions into 10 bins (deciles), and in each bin
compare the average prediction with the actual share of drops. A calibrated
model's points fall on the diagonal.

Calibration matters here because the expected-value rule (section 11) uses
$p$ directly. If $p$ is systematically off, the cutoff is off too.

---

## 16. Shapley values (SHAP) *(notebook 06)*

SHAP splits one prediction into contributions from each input, treating the
inputs like players sharing a payout (from cooperative game theory). Input
$j$'s contribution is its marginal effect, averaged over every order in which
the inputs could be added:

$$\phi_j = \sum_{S \subseteq N \setminus \{j\}}
\frac{|S|!\,(n - |S| - 1)!}{n!}\,\Big[v(S \cup \{j\}) - v(S)\Big]$$

- $N$ is the set of all $n$ inputs, and $S$ is a subset that doesn't include
  $j$.
- $v(S)$ is the model's expected output when only the inputs in $S$ are
  known.
- The fraction is the share of orderings in which exactly the inputs in $S$
  come before $j$.

**Key property (efficiency).** The contributions add up exactly to how far
this prediction is from the average prediction:

$$\sum_j \phi_j = f(x) - \mathbb{E}[f(x)]$$

For LightGBM, these are in log-odds. For tree models they can be computed
exactly and quickly (TreeSHAP), which LightGBM does with `pred_contrib=True`.

---

## 17. The bootstrap and percentile confidence intervals *(notebook 09)*

I have one sample of trips and one estimate from it, for example the average
% saved, $\hat\theta$. A confidence interval says how much $\hat\theta$ could
move with a different sample. I can't collect new samples, so the bootstrap
**simulates** them by resampling the data I have:

1. Draw $n$ trips **with replacement** from my $n$ trips. Some appear twice,
   some not at all.
2. Recompute the estimate on that resample: $\hat\theta^{*}_b$.
3. Repeat $B$ times ($B = 2{,}000$).

The spread of the $\hat\theta^{*}_b$ approximates the sampling spread of
$\hat\theta$. The **95% percentile interval** is the middle 95% of the draws:

$$\Big[\, \hat\theta^{*}_{(2.5\%)},\ \hat\theta^{*}_{(97.5\%)} \,\Big]$$

**Why it works.** The real data come from an unknown distribution $F$. The
bootstrap treats the sample's own distribution $\hat F$ as a stand-in for
$F$, and resampling from $\hat F$ mimics drawing new samples from $F$. It needs
no formula for the estimate's standard error, which is handy for things like
"total % saved" (a ratio of sums).

**Monte Carlo error.** The interval's endpoints are themselves estimated from
$B$ random draws, so they wobble a little from run to run. That wobble shrinks
like $1/\sqrt{B}$. With $B = 2{,}000$ it's small compared with the width of the
interval.

---

## 18. The cluster bootstrap *(notebook 09)*

The ordinary bootstrap assumes the $n$ trips are **independent**. They
aren't: trips on the same route share airlines, competition and demand, so
their results move together.

**Why that matters.** Suppose trip $i$'s result is a route effect plus noise,
$r_i = u_{\text{route}(i)} + e_i$, with $\operatorname{Var}(u) = \sigma_u^2$ and
$\operatorname{Var}(e) = \sigma_e^2$. With $R$ routes of $m$ trips each
($n = Rm$), the variance of the overall average is

$$\operatorname{Var}(\bar r) = \frac{\sigma_u^2}{R} + \frac{\sigma_e^2}{n}$$

The route part shrinks with the number of **routes**, not the number of
trips. Treating trips as independent effectively assumes
$\operatorname{Var}(\bar r) = (\sigma_u^2 + \sigma_e^2)/n$, which is much too
small when $\sigma_u^2$ matters and $m$ is large.

**The fix.** Resample **whole routes**: draw $R$ routes with replacement and
keep all of each route's trips together. Then every resample reflects
route-to-route variation the way the real data does.

In notebook 09 this made the intervals **2–3× wider** than the ordinary
bootstrap's. The estimate didn't change, only the honesty about its
uncertainty.

**Interview version:** "Trips on the same route are correlated, so I used a
cluster bootstrap by route. The effective sample size is closer to the number
of routes than the number of trips, and the honest intervals came out 2–3×
wider."

---

## 19. Paired comparison *(notebook 09)*

To compare the model with a simple rule, I use the **same trips** for both
and look at the per-trip difference:

$$d_i = r_i^{\text{model}} - r_i^{\text{rule}}, \qquad \bar d = \frac{1}{n}\sum_i d_i$$

**Why pairing helps.** The variance of a difference is

$$\operatorname{Var}(r^{\text{model}} - r^{\text{rule}}) = \operatorname{Var}(r^{\text{model}}) + \operatorname{Var}(r^{\text{rule}}) - 2\operatorname{Cov}(r^{\text{model}}, r^{\text{rule}})$$

On the same trips the two results are positively correlated (a route where
fares fall helps both), so the covariance term is large and the variance of
$d_i$ is small. Comparing two separate averages would ignore that and give a
much wider interval.

I bootstrap $\bar d$ by route (section 18). If the 95% interval excludes zero,
the difference isn't plausibly due to chance. From 30 days out the model beats
"buy 21 days out" by 6.1 points (CI 5.2–7.0). From 60 days out it loses by 6.6
points (CI 4.9–8.3).

---

## 20. Estimating a Markov chain *(notebook 07)*

The state $x$ (today's fare relative to the route's usual fare) is placed on
a grid of $K$ points. A **Markov chain** says tomorrow's grid point depends
only on today's: $P(x_{t-1} = j \mid x_t = i) = M_{ij}$.

**Maximum-likelihood estimate.** If $n_{ij}$ is the number of observed moves
from $i$ to $j$, the likelihood of all observed moves is
$\prod_{i,j} M_{ij}^{\,n_{ij}}$. Maximize its log subject to each row summing
to 1, using a Lagrange multiplier $\nu_i$ per row:

$$\frac{\partial}{\partial M_{ij}}\Big[\sum_{j} n_{ij}\log M_{ij} - \nu_i\Big(\sum_j M_{ij} - 1\Big)\Big]
= \frac{n_{ij}}{M_{ij}} - \nu_i = 0
\;\Longrightarrow\; M_{ij} = \frac{n_{ij}}{\nu_i}$$

The row constraint forces $\nu_i = \sum_j n_{ij}$, so

$$\hat M_{ij} = \frac{n_{ij}}{\sum_k n_{ik}}$$

The estimate is just the share of moves out of $i$ that went to $j$. I pool
nearby days for stability, and add a small pseudo-count to "stay put" so that
rows with almost no data default to "no change".

---

## 21. The jump model and its maximum-likelihood fit *(notebook 07)*

Each day, the fare either stays put or jumps:

$$x_{t-1} = x_t + B_t\, J_t, \qquad B_t \sim \text{Bernoulli}(\lambda), \qquad
J_t \mid x_t \sim \text{Normal}(\mu + \kappa x_t,\ \sigma^2)$$

$\kappa < 0$ means **pull-back**: a fare above its usual level ($x > 0$)
tends to jump down.

**The likelihood splits into two independent parts.** Each observed move is
either "no change" (probability $1-\lambda$) or "a jump of size $\Delta$"
(probability $\lambda$ times the normal density of $\Delta$). So

$$\log L = \underbrace{n_1\log\lambda + n_0\log(1-\lambda)}_{\text{depends only on }\lambda}
\;+\; \underbrace{\sum_{\text{jumps}} \log \phi\!\left(\frac{\Delta_i - \mu - \kappa x_i}{\sigma}\right) - n_1\log\sigma}_{\text{depends only on }\mu,\kappa,\sigma}$$

where $n_1$ is the number of jumps and $n_0$ the number of no-change days.
Each part can be maximized on its own:

- **$\lambda$:** setting $\frac{n_1}{\lambda} - \frac{n_0}{1-\lambda} = 0$
  gives $\hat\lambda = \frac{n_1}{n_0 + n_1}$, the share of days with a jump.
- **$\mu, \kappa$:** maximizing a normal log-likelihood in the mean is the
  same as minimizing $\sum (\Delta_i - \mu - \kappa x_i)^2$, which is
  **ordinary least squares** on the jump days.
- **$\sigma^2$:** setting the derivative to zero gives the mean squared
  residual, $\hat\sigma^2 = \frac{1}{n_1}\sum(\Delta_i - \hat\mu - \hat\kappa x_i)^2$.

A unit test simulates data with known $\lambda = 0.5$, $\kappa = -0.3$ and
$\sigma = 0.1$ and checks that the fit recovers them.

**On the grid.** From grid point $i$, the probability of landing in bin
$[e_j, e_{j+1})$ after a jump is
$\Phi\!\big(\frac{e_{j+1} - m_i}{\sigma}\big) - \Phi\!\big(\frac{e_j - m_i}{\sigma}\big)$,
with $m_i = x_i + \mu + \kappa x_i$ and $\Phi(z) = \tfrac12\operatorname{erfc}(-z/\sqrt2)$.
The no-jump probability $1 - \lambda$ is added to staying at $i$.

---

## 22. Backward induction on a grid, and convergence *(notebook 07)*

On the grid, the Bellman equation (section 6) becomes matrix algebra. With
$\mathbf{c}$ the cost at each grid point and $M_t$ the transition matrix:

$$\mathbf{V}_1 = \mathbf{c}, \qquad \mathbf{C}_t = M_t\,\mathbf{V}_{t-1}, \qquad
\mathbf{V}_t = \min(\mathbf{c},\ \mathbf{C}_t) \text{ (element by element)}$$

Each day is one $K \times K$ matrix-vector product, so the total cost is
$O(T K^2)$ for $T$ days.

**What "cost" means.** With $\mathbf{c} = e^{x}$ the solver minimizes the
expected fare (dollars). With $\mathbf{c} = x$ it minimizes the expected log
fare, and since $\log P_\tau - \log P_s$ is approximately the percent change,
that approximately targets average %. Notebook 07 tries both.

**Discretization error.** Each grid point stands in for a whole bin of width
$h = (x_{\max} - x_{\min})/K$. Representing a bin by its midpoint makes the
leading error terms cancel, so the error shrinks like $h^2$:

$$V^{(K)} = V + a\,h^2 + O(h^3)$$

**Measuring the order.** Doubling $K$ halves $h$, so successive changes
$\Delta_K = |V^{(2K)} - V^{(K)}|$ shrink by a factor of $2^p$ for order $p$:

$$p \approx \log_2\frac{\Delta_K}{\Delta_{2K}}$$

Notebook 07 measures $p \approx 1.7$–$2.1$, consistent with second order.

---

## 23. Monte Carlo error *(notebook 07)*

To check the DP answer independently, I simulate $N$ price paths, follow the
policy on each, and average the price paid: $\bar Y_N = \frac1N\sum_i Y_i$.

By the central limit theorem, for large $N$,

$$\bar Y_N \approx \text{Normal}\Big(\mathbb{E}[Y],\ \frac{\operatorname{Var}(Y)}{N}\Big),
\qquad \text{standard error} = \frac{s}{\sqrt N}$$

where $s$ is the sample standard deviation. So the error shrinks like
$1/\sqrt N$: **100× more paths for 10× more accuracy**. On a log-log plot that's
a line of slope $-\tfrac12$. The simulated mean should land within about 2
standard errors of the DP value, and it does at every $N$ in notebook 07.

---

## 24. Interpolation and kinks *(notebook 07)*

The policy needs $C_t(x)$ between grid points.

- **Linear interpolation** has error $O(h^2)$ when the function has a
  bounded second derivative, and it never overshoots.
- **Cubic methods** (like pchip) reach $O(h^3)$ or better, **but only if the
  function is smooth**.

The value function $V_t(x) = \min\{\text{cost}(x),\ C_t(x)\}$ has a **kink**
where the two are equal: its slope jumps from the slope of the cost to the
slope of $C_t$. Near a kink a cubic assumes a smoothness that isn't there, so
its accuracy advantage disappears. In notebook 07, linear interpolation gave
the wrong buy/wait decision 0.25% of the time and pchip 1.8%.

---

## 25. The value of waiting (real options) *(notebook 07)*

With $t$ days left and today's fare at the route's usual level ($x = 0$, cost
1), buying now costs 1 and waiting optimally costs $V_t(0) \le 1$. The
**value of the right to wait** is

$$O_t = 1 - V_t(0) = \max\{0,\ 1 - C_t(0)\} \ge 0$$

This is the same structure as an American option: the holder can "exercise"
(buy) at any time, and the option's value is the expected gain from choosing
the best time instead of exercising immediately. As in option pricing, $O_t$
shrinks as expiry (departure) approaches and hits zero once immediate
exercise is optimal, which is about 14 days out here.

Like an option priced with the wrong volatility, $O_t$ is only as good as the
price model: the model says waiting is worth 19% from 60 days out, while the
backtest realized about 6%.

---

## 26. Fixed effects and the within transformation *(notebook 03)*

The booking-curve regression is

$$\log P_{it} = \alpha_i + \sum_{d} \beta_d\, \mathbf{1}[t = d] + \varepsilon_{it}$$

with a separate intercept $\alpha_i$ for each of about 50,000 trips. Estimating
50,000 dummy variables directly would be slow, but there's a shortcut.

**The within transformation.** Average the equation over trip $i$'s
observations, and subtract that average from each observation:

$$\log P_{it} - \overline{\log P_i} = \sum_d \beta_d\Big(\mathbf{1}[t=d] - \overline{\mathbf{1}[t=d]}_i\Big) + (\varepsilon_{it} - \bar\varepsilon_i)$$

The $\alpha_i$ cancels, because it's the same for every observation of the
trip. Running ordinary least squares on the demeaned data gives exactly the
same $\hat\beta$ as including all the dummies (the Frisch–Waugh–Lovell
theorem).

**What it buys.** Anything constant within a trip (route, flight date,
season, overall demand) is absorbed by $\alpha_i$ and can't bias $\beta_d$. So
$\beta_d$ compares each trip **only with itself** at different days to
departure.

**What it can't do.** It can't estimate effects of things that are constant
within a trip, like the flight's day of week. That's why the flight-day
regression uses route and month fixed effects instead.

---

## 27. Clustered standard errors *(notebook 03)*

The usual OLS standard errors assume every error $\varepsilon_{it}$ is
independent. On the same route, they aren't. The **cluster-robust** variance
estimator allows any correlation within a cluster $g$ (a route) but none
between clusters:

$$\widehat{\operatorname{Var}}(\hat\beta) = (X^\top X)^{-1}\Big(\sum_{g} X_g^\top \hat e_g\, \hat e_g^\top X_g\Big)(X^\top X)^{-1}$$

where $X_g$ and $\hat e_g$ are the rows and residuals for cluster $g$. It's
called a **sandwich** estimator: the two outer terms are the "bread", and
the middle term is the "meat" that captures within-cluster correlation.

It's the analytic counterpart of the cluster bootstrap (section 18). Both
make the effective sample size depend on the number of routes, not the number
of rows. With about 235 clusters, the large-sample approximation is
reasonable.

---

## 28. Regression discontinuity at the deadlines *(notebook 03)*

If a cheap fare class closes when the 21-day deadline passes, the fare
should jump **right at** the cutoff, beyond whatever smooth trend it was
already on. The test regresses the daily change in log fare on:

$$\Delta\log P_{it} = \gamma_{21}D^{21}_t + \gamma_{14}D^{14}_t + \gamma_{7}D^{7}_t + f(t) + \alpha_{\text{route}} + \varepsilon_{it}$$

- $D^{k}_t = 1$ on the first day after deadline $k$ has passed.
- $f(t)$ is a cubic polynomial in days to departure, which models the smooth
  climb.

$\gamma_k$ is the **discontinuity**: the jump at the cutoff that the smooth
trend can't explain. The identifying assumption is that nothing else changes
abruptly at exactly that day. Estimates: $\gamma_{21} = 0.050$,
$\gamma_{14} = 0.097$, $\gamma_{7} = 0.022$, with clustered standard errors of
0.004–0.006.

---

## 29. Reading log coefficients, and the HHI *(notebook 03)*

**Log coefficients.** In $\log P = \beta x + \dots$, increasing $x$ by 1
multiplies the fare by $e^{\beta}$, so the percent change is

$$e^{\beta} - 1 \;\approx\; \beta \quad\text{for small } \beta$$

For the Spirit/Frontier dummy, $\beta = -0.446$ gives
$e^{-0.446} - 1 = -36\%$, noticeably different from the naive reading of
−44.6%. For log distance, $\beta = 0.37$ is an **elasticity**: 1% more distance
gives about 0.37% higher fares.

**The Herfindahl–Hirschman index.** With market shares $s_a$ summing to 1,

$$\text{HHI} = \sum_a s_a^2$$

A monopoly gives 1. With $n$ equal firms each $s_a = 1/n$, so
$\text{HHI} = n \cdot (1/n)^2 = 1/n$. Unequal shares push HHI up, because
squaring weights large shares more. Here the shares are of **listed
options**, a proxy, since the data has no passenger counts.

---

## 30. Autocorrelation and the variance ratio *(notebook 03)*

Let $r_t = \log P_t - \log P_{t+1}$ be the daily change. For a **random
walk**, changes are uncorrelated, and the variance of a $q$-day change adds
up:

$$\operatorname{Var}\Big(\sum_{k=0}^{q-1} r_{t+k}\Big) = \sum_{k} \operatorname{Var}(r_{t+k}) = q\,\sigma^2$$

In general, the covariance terms don't vanish:

$$\operatorname{Var}\Big(\sum_{k=0}^{q-1} r_{t+k}\Big) = q\sigma^2 + 2\sum_{j=1}^{q-1}(q-j)\,\operatorname{Cov}(r_t, r_{t+j})$$

Dividing by $q\sigma^2$:

$$\text{VR}(q) = 1 + 2\sum_{j=1}^{q-1}\Big(1 - \frac{j}{q}\Big)\rho_j$$

where $\rho_j$ is the autocorrelation at lag $j$. **Negative autocorrelation
gives VR < 1**: moves cancel out over time (mean reversion). With
$\rho_1 \approx -0.28$ alone, $\text{VR}(2) = 1 + \rho_1 = 0.72$, which is
exactly what notebook 03 measures. The ratios keep falling at longer
horizons (0.23 at 20 days), so reversal continues beyond one day.

---

## 31. Fitted Q-iteration *(notebook 08)*

**The value of waiting.** Let $W_t(s)$ be the expected cost of waiting when
I'm $t$ days out in state $s$ (everything I know that day), and then acting
optimally. If the next price I see is on day $u < t$:

$$W_t(s) = \mathbb{E}\Big[\min\big\{\text{cost}(x_u),\ W_u(s_u)\big\} \;\Big|\; s_t = s\Big],
\qquad W_u \text{ replaced by } \infty \text{ on the last day}$$

This is the Bellman equation (section 6) written for the "wait" action. In
reinforcement-learning terms, $W_t(s)$ is the **Q-value** of waiting, and the
Q-value of buying is just $\text{cost}(x_t)$.

**Estimating an expectation by regression.** A conditional expectation
$\mathbb{E}[Y \mid s]$ is the function $g$ that minimizes expected squared
error:

$$\mathbb{E}[Y \mid s] = \arg\min_g\ \mathbb{E}\big[(Y - g(s))^2\big]$$

So fitting a regression of the realized targets
$y = \min\{\text{cost}(x_u), \hat W_u(s_u)\}$ on $s$ estimates $W_t$, with no
price model. Here the regression is LightGBM.

**Why the order works.** The target for day $t$ only involves days $u < t$.
Fitting $\hat W_2$ first (whose targets need no $\hat W$ at all, since day 1
is a forced buy), then $\hat W_3$, and so on up to $\hat W_{60}$, every target
is available when it's needed. That's backward induction in time (section
22), with each expectation estimated from data instead of computed from a
transition matrix.

**Policy.** Buy on day $t$ if $\text{cost}(x_t) \le \hat W_t(s_t)$.

**Compared with the other approaches:**

| Approach | How it estimates the value of waiting | Weak spot |
|---|---|---|
| Dynamic programming (07) | Calculated from an assumed price model | Wrong if the model is wrong |
| LightGBM classifier (06) | Doesn't. It predicts "will it drop 5% soon?" and uses a cutoff | Optimizes a stand-in, not the price paid |
| Fitted Q-iteration (08) | Regression on realized outcomes over the whole remaining window | Needs plenty of data for every day; inherits the data's season |

**One known bias.** Using a fitted $\hat W_u$ inside $\min\{\cdot\}$ makes the
targets slightly optimistic, because the minimum of a noisy estimate tends to
be too low (the same issue behind "double Q-learning"). The backtest doesn't
share this bias, since it scores the actual prices paid.